# 02. 날씨 패턴 분석 (Weather Patterns)

## 목표
- 기상 데이터와 이용량 병합
- 강수량, 기온, 풍속, 습도와 이용량의 상관관계 분석
- 강수 구간별 이용 패턴 분석

## ULTRA-THINK Framework: L - Look Deeper (심층 탐색)

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.data_loader import load_bike_data, load_weather_data
from src.utils.preprocessing import resample_to_hourly, merge_weather_data, create_derived_features
from src.utils.visualization import plot_rain_bins, plot_hourly_usage

import warnings
warnings.filterwarnings('ignore')

## 1. 데이터 로딩 및 전처리

In [ ]:
# 데이터 로딩
bike = load_bike_data(data_dir="../data/raw")
weather = load_weather_data(file_path="../data/external/서울시 지상관측자료 정보(일별).csv")

In [ ]:
# 5분 → 1시간 리샘플링
hourly = resample_to_hourly(bike)

In [ ]:
# 기상 데이터 병합
merged = merge_weather_data(hourly, weather)
merged.head()

In [ ]:
# 파생 변수 생성
df = create_derived_features(merged)
df.head()

## 2. 강수량과 이용량의 관계

In [ ]:
# 강수 구간별 평균 이용량
plot_rain_bins(df, save_path="../outputs/figures/bar_rain_bins.png")

In [ ]:
# 강수 구간별 통계
rain_stats = df.groupby('강수_구간', observed=True)['전체_건수'].agg(['mean', 'median', 'std', 'count'])
print("\n강수 구간별 이용 통계:")
print(rain_stats)

## 3. 기상 변수 간 상관관계

In [ ]:
# 상관관계 행렬
corr_cols = ['전체_건수', '평균기온(℃)', '일강수량(mm)', '평균풍속(m/s)', '평균상대습도(%)']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('기상 변수와 이용량 상관관계', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../outputs/figures/corr_weather_usage.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. 맑은 날 vs 비 오는 날 비교

In [ ]:
# 맑은 날 vs 비 오는 날 평균 이용량
clear_avg = df[df['비여부'] == 0]['전체_건수'].mean()
rainy_avg = df[df['비여부'] == 1]['전체_건수'].mean()
decrease_pct = ((rainy_avg - clear_avg) / clear_avg) * 100

print(f"\n맑은 날 평균 이용량: {clear_avg:.2f} 건")
print(f"비 오는 날 평균 이용량: {rainy_avg:.2f} 건")
print(f"감소율: {decrease_pct:.2f}%")

In [ ]:
# 시각화
fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(['맑은 날', '비 오는 날'], [clear_avg, rainy_avg], color=['#FDB462', '#80B1D3'])
ax.set_ylabel('평균 이용 건수', fontsize=12)
ax.set_title('맑은 날 vs 비 오는 날 이용량 비교', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# 값 표시
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/clear_vs_rainy.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. 시간대별 이용량 추이

In [ ]:
plot_hourly_usage(df, save_path="../outputs/figures/line_hourly_usage.png")

## 6. 데이터 저장

In [ ]:
# 전처리된 데이터 저장
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/merged_hourly_202509.csv", index=False, encoding="utf-8")
print("✅ 전처리된 데이터 저장 완료: data/processed/merged_hourly_202509.csv")

## 7. 다음 단계

- ✅ 기상 데이터 병합 완료
- ✅ 날씨와 이용량의 관계 분석 완료
- 다음: `03_time_space_analysis.ipynb`에서 시공간 분석